# Neue Laufdaten: Artikelstatistik und Grafiken

Eingaben: ein explizites Laufverzeichnis mit `data/summaries.csv`, optional `answers.csv` und `metrics_detailed.csv`.
Frischer Kernel: `python 05_analyze.py --run LAUF --output results/runs/neue-auswertung`.
Ausgaben: `AUSGABE/analysis/`. Keine Modellaufrufe. Humanratings, historische Versionsvergleiche und Panelregressionen gelten ausschließlich für den historischen Bestand.
Artikelbootstrap: 10.000 Ziehungen, Seed 32. Fehlende QA und ausgelassene Modellmetriken bleiben getrennt.

In [ ]:
import os
from pathlib import Path
from faktenretention.offline_guard import install
install()
from faktenretention.analysis_data import analyze_run
RUN = Path(os.environ['FAKTENRETENTION_NEW_RUN']).resolve()
OUTPUT = Path(os.environ['FAKTENRETENTION_NEW_OUTPUT']).resolve()
assert RUN != OUTPUT and RUN not in OUTPUT.parents
output = analyze_run(RUN, output=OUTPUT)

## Artikelmittel und Unsicherheit
Nur im gewählten Lauf verfügbare Metriken werden ausgewertet.

In [ ]:
import pandas as pd
from IPython.display import display
from faktenretention.notebook_context import display_figure
plotted = pd.read_csv(output/'mittelwerte_95ci.csv')
display(plotted.head(24))

## Grafiken
Punktweise 95-%-Perzentilintervalle; stabile Methodenfarben und Reihenfolge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from faktenretention.protocol import STAGES, METHODS, LABELS
COLORS = dict(zip(METHODS, ['#777777','#0072B2','#E69F00','#009E73','#CC79A7','#D55E00']))
for metric in ['retention_observed','retention_adjusted','D']:
    sub = plotted[plotted.metric.eq(metric)]
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(7,4))
    for method, group in sub.groupby('method',sort=False):
        group = group.set_index('stage_target_words').reindex([s for s in STAGES if s in set(group.stage_target_words)])
        ax.errorbar(group.index.astype(str),group['mean'],
            yerr=np.maximum(0,np.array([group['mean']-group.ci_low,group.ci_high-group['mean']])),
            marker='o',color=COLORS[method],label=LABELS[method])
    ax.set(xlabel='Zielwörter',ylabel=metric)
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(output/(metric+'.png'),dpi=160)
    display_figure(fig)
    plt.close(fig)